In [ ]:
!apt-get install -y git
!pip install uv

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
!git clone https://github.com/Flora-jia-jfr/CPS572_Sp26_mini_project.git /content/project
%cd /content/project

fatal: destination path '/content/project' already exists and is not an empty directory.
/content/project


In [ ]:
!uv pip install -r requirements.txt --system

Using Python 3.12.13 environment at: /usr
Resolved 142 packages in 3.27s
Checked 142 packages in 13ms


In [ ]:
import os
os.environ["TINKER_API_KEY"] = "tml-4UOPjGEJmlgIt4YbrrT4zNF0vkJGplzptnOe5p6qPRkCel5X7vDMaNwWmmgySX6EDAAAA"
os.environ["PYTHONPATH"] = "/content/project"

In [ ]:
!python evaluation/eval_all.py --base_models meta-llama/Llama-3.2-3B --limit 5
!python evaluation/eval_all.py --base_models meta-llama/Llama-3.2-3B
!python evaluation/eval_all.py --base_models meta-llama/Llama-3.2-1B meta-llama/Llama-3.2-3B meta-llama/Llama-3.1-8B


############################################################
# BASELINE (core): meta-llama/Llama-3.2-3B
############################################################
INFO:__main__:Evaluating: base:meta-llama/Llama-3.2-3B
INFO:__main__:Renderer: role_colon | temperature=0.0 top_p=1.0
INFO:__main__:============================================================
INFO:__main__:TASK 1/3: IFEval (Instruction Following)
INFO:__main__:============================================================
INFO:evaluation.eval_ifeval:Model: meta-llama/Llama-3.2-3B | Renderer: role_colon
INFO:tinker.lib.public_interfaces.service_client:ServiceClient initialized for session 337a080f-fe7d-5d34-a6b0-d1feb0eba566
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
tokenizer_config.json: 55.4kB [00:00, 9.07MB/s]
tokenizer.json: 9.09MB [00:00, 83.8MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.04MB/s]
INFO:datasets:TensorFlow version 2.19.0 available.
INFO:datasets:JAX version 0.7.2 available.
Loading 

In [ ]:
%%writefile evaluation/train_and_publish.py

import argparse
import json
import math
import os
import re
import ast
from collections import Counter, defaultdict

import numpy as np
import tinker
from tinker import types
from tinker_cookbook import model_info, renderers
from tinker_cookbook.supervised.data import conversation_to_datum
from tinker_cookbook.tokenizer_utils import get_tokenizer

MODEL = "meta-llama/Llama-3.1-8B"
EVAL_DIR = os.path.dirname(os.path.abspath(__file__))
TASK_NAMES = {0: "GSM8K", 1: "IFEval", 2: "Code"}


def get_lr(step, base_lr, warmup_steps, total_steps, min_lr_ratio=0.1):
    if step < warmup_steps:
        return base_lr * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    cosine = 0.5 * (1 + math.cos(math.pi * progress))
    return base_lr * (min_lr_ratio + (1 - min_lr_ratio) * cosine)


def _extract_last_number(text):
    nums = re.findall(r"-?\d[\d,]*\.?\d*", text.replace(",", ""))
    return nums[-1] if nums else None


def reward_gsm8k(generated, reference):
    score = 0.0
    pred = _extract_last_number(generated)
    gold = _extract_last_number(reference)
    if pred is not None and gold is not None and pred == gold:
        score = 1.0
    if "####" in generated:
        score = min(score + 0.1, 1.0)
    return score


def parse_constraints_from_instruction(instruction: str):
    constraints = []
    text = instruction.lower()

    m = re.search(r'exactly\s+(\d+)\s+words?', text)
    if m:
        n = int(m.group(1))
        constraints.append({"type": "max_words", "value": n + 5})
        constraints.append({"type": "min_words", "value": n - 5})

    m = re.search(r'(?:in|use|within)\s+(\d+)\s+words?', text)
    if m and not constraints:
        constraints.append({"type": "max_words", "value": int(m.group(1))})

    m = re.search(r'(\d+)\s+sentences?', text)
    if m:
        constraints.append({"type": "sentence_count", "value": int(m.group(1))})

    m = re.search(r'(\d+)\s+paragraphs?', text)
    if m:
        constraints.append({"type": "paragraph_count", "value": int(m.group(1))})

    for m in re.finditer(r'include\s+(?:the\s+)?(?:word|phrase)\s+["\']([^"\']+)["\']', text):
        constraints.append({"type": "keyword_include", "value": m.group(1)})

    for m in re.finditer(r'(?:do not|avoid)\s+(?:use\s+)?(?:the\s+)?word\s+["\']([^"\']+)["\']', text):
        constraints.append({"type": "keyword_exclude", "value": m.group(1)})

    if re.search(r'bullet\s*points?|bulleted\s*list', text):
        constraints.append({"type": "has_bullets"})
    if re.search(r'numbered\s*list', text):
        constraints.append({"type": "has_numbered_list"})

    if re.search(r'\ball\s+caps?\b|\buppercase\b', text):
        constraints.append({"type": "all_caps"})

    return constraints


def reward_ifeval(generated, constraints):
    if not constraints:
        return _reward_ifeval_proxy(generated)

    satisfied = 0
    words = generated.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', generated) if s.strip()]

    for c in constraints:
        ctype = c.get("type", "")
        val   = c.get("value")

        try:
            if   ctype == "keyword_include":
                satisfied += int(str(val).lower() in generated.lower())

            elif ctype == "keyword_exclude":
                satisfied += int(str(val).lower() not in generated.lower())

            elif ctype == "max_words":
                satisfied += int(len(words) <= int(val))

            elif ctype == "min_words":
                satisfied += int(len(words) >= int(val))

            elif ctype == "sentence_count":
                satisfied += int(abs(len(sentences) - int(val)) <= 1)

            elif ctype == "paragraph_count":
                paras = [p for p in generated.split("\n\n") if p.strip()]
                satisfied += int(abs(len(paras) - int(val)) <= 1)

            elif ctype == "has_bullets":
                satisfied += int(bool(re.search(r'^\s*[-*•]\s', generated, re.MULTILINE)))

            elif ctype == "has_numbered_list":
                satisfied += int(bool(re.search(r'^\s*\d+\.\s', generated, re.MULTILINE)))

            elif ctype == "all_caps":
                alpha = re.sub(r'[^a-zA-Z]', '', generated)
                satisfied += int(alpha and alpha == alpha.upper())

        except:
            pass

    return satisfied / len(constraints)


def _reward_ifeval_proxy(generated: str) -> float:
    r = 0.0
    words = generated.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', generated) if s.strip()]

    if len(sentences) >= 3:
        r += 0.25
    if 50 <= len(words) <= 400:
        r += 0.25
    if len(sentences) >= 2 and len(set(sentences))/max(len(sentences),1) > 0.7:
        r += 0.25
    if re.search(r'(^\s*[-*•]\s|\d+\.\s|#{1,3}\s)', generated, re.MULTILINE):
        r += 0.25

    return r


def reward_code(generated, reference):
    code_match = re.search(r"```(?:python)?\s*(.*?)```", generated, re.DOTALL)
    if code_match:
        try:
            ast.parse(code_match.group(1))
            return 1.0
        except SyntaxError:
            return 0.5
    ref_ids = set(re.findall(r"\b[a-zA-Z_]\w*\b", reference))
    gen_ids = set(re.findall(r"\b[a-zA-Z_]\w*\b", generated))
    overlap = len(ref_ids & gen_ids) / max(len(ref_ids), 1)
    return 0.2 if overlap > 0.3 else 0.0


def compute_reward(example):

    task_id  = example.get("task_id", -1)
    messages = example.get("messages", [])
    generated = messages[-1]["content"] if messages else ""

    if task_id == 0:
        return reward_gsm8k(generated, example.get("reference_answer", ""))
    elif task_id == 1:
        return reward_ifeval(generated, example.get("constraints", []))
    elif task_id == 2:
        return reward_code(generated, example.get("reference_answer", ""))
    return 0.5


def normalise_rewards(rewards, eps=1e-8):

    r = np.array(rewards, dtype=np.float32)
    rng = r.max() - r.min()
    if rng < eps:
        return [1.0] * len(rewards)
    normed = (r - r.min()) / rng
    return (0.2 + 0.8 * normed).tolist()


def batch_contrastive_scale(rewards, clip=0.3, eps=1e-8):

    r = np.array(rewards, dtype=np.float32)
    advantages = (r - r.mean()) / (r.std() + eps)
    scales = np.clip(1.0 + advantages, 1.0 - clip, 1.0 + clip)
    return scales.tolist()


def apply_weight_scale(datum, scale):

    old_w = datum.loss_fn_inputs["weights"].to_numpy()
    datum.loss_fn_inputs["weights"] = types.TensorData.from_numpy(
        old_w * scale
    )
    return datum


class TaskLRAdapter:

    def __init__(self, task_ids, window=200, boost=0.25):
        self.window = window
        self.boost  = boost
        self._buf   = []
        self._means = {tid: 0.5 for tid in task_ids}   # neutral start

    def update(self, task_id, reward):
        self._buf.append((task_id, reward))
        if len(self._buf) > self.window:
            self._buf.pop(0)
        by_task = defaultdict(list)
        for tid, r in self._buf:
            by_task[tid].append(r)
        for tid, rs in by_task.items():
            self._means[tid] = float(np.mean(rs))

    def scale(self, task_id):
        m = self._means.get(task_id, 0.5)
        return 1.0 + self.boost * (0.5 - m) / 0.5


class ShuffledDataLoader:
    def __init__(self, data, batch_size, seed=42):
        self.data       = data
        self.batch_size = batch_size
        self.rng        = np.random.default_rng(seed)
        self.indices    = np.arange(len(self.data))
        self.rng.shuffle(self.indices)
        self.pos        = 0
        self.epoch      = 0

    def next_batch(self):
        if self.pos + self.batch_size > len(self.indices):
            self.rng.shuffle(self.indices)
            self.pos = 0
            self.epoch += 1
        idx = self.indices[self.pos: self.pos + self.batch_size]
        self.pos += self.batch_size
        return [self.data[i] for i in idx]


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_steps",        type=int,   default=3000)
    parser.add_argument("--batch_size",       type=int,   default=4)
    parser.add_argument("--lr",               type=float, default=1e-4)
    parser.add_argument("--rank",             type=int,   default=32)
    parser.add_argument("--warmup_steps",     type=int,   default=100)
    parser.add_argument("--max_length",       type=int,   default=1024)
    parser.add_argument("--checkpoint_name",  type=str,   default="multitask_rl_v3")
    parser.add_argument("--data_path",        type=str,   default="training_data.jsonl")
    parser.add_argument("--no_publish",       action="store_true")
    parser.add_argument("--seed",             type=int,   default=42)
    parser.add_argument("--log_interval",     type=int,   default=50)
    parser.add_argument("--save_every",       type=int,   default=500)
    parser.add_argument("--intermediate_ttl", type=int,   default=86400)
    parser.add_argument("--resume_from",      type=str,   default=None)
    parser.add_argument("--sft_steps",        type=int,   default=1000)
    parser.add_argument("--rl_lr_scale",      type=float, default=0.3)
    parser.add_argument("--use_reward_weights",  action="store_true", default=True,
                        help="Scale token loss by normalised per-example reward")
    parser.add_argument("--no_reward_weights",   dest="use_reward_weights", action="store_false")
    parser.add_argument("--use_batch_contrast",  action="store_true", default=True,
                        help="Apply within-batch contrastive reward scaling (no sampling needed)")
    parser.add_argument("--no_batch_contrast",   dest="use_batch_contrast", action="store_false")
    parser.add_argument("--use_task_lr",         action="store_true", default=True,
                        help="Adaptively scale LR per task based on recent reward")
    parser.add_argument("--no_task_lr",          dest="use_task_lr", action="store_false")
    parser.add_argument("--contrast_clip",       type=float, default=0.3,
                        help="PPO-clip bound for batch-contrastive scaling")
    parser.add_argument("--task_lr_boost",       type=float, default=0.25,
                        help="Max LR boost/reduction for adaptive task LR")
    parser.add_argument("--task_lr_window",      type=int,   default=200,
                        help="Rolling window (steps) for task reward tracking")
    args = parser.parse_args()

    np.random.seed(args.seed)
    print("=" * 65)
    print(f"Model:              {MODEL}")
    print(f"Steps:              {args.num_steps}  (SFT warm-start: {args.sft_steps})")
    print(f"Batch size:         {args.batch_size}")
    print(f"LR:                 {args.lr}  (phase-2 × {args.rl_lr_scale})")
    print(f"LoRA rank:          {args.rank}")
    print(f"Reward weights:     {args.use_reward_weights}")
    print(f"Batch contrast:     {args.use_batch_contrast}  (clip={args.contrast_clip})")
    print(f"Task adaptive LR:   {args.use_task_lr}  (boost={args.task_lr_boost})")
    print("=" * 65)

    tokenizer     = get_tokenizer(MODEL)
    renderer_name = model_info.get_recommended_renderer_name(MODEL)
    renderer      = renderers.get_renderer(renderer_name, tokenizer)
    print(f"Renderer: {renderer_name}\n")

    print("Loading data...")
    with open(args.data_path) as f:
        raw_data = [json.loads(l) for l in f]
    raw_data = [ex for i, ex in enumerate(raw_data) if i != 12452]
    print(f"  {len(raw_data)} examples")

    dist = Counter(ex["task_id"] for ex in raw_data)
    for tid, name in TASK_NAMES.items():
        print(f"  {name}: {dist.get(tid, 0)}")

    print("\nPre-computing rewards...")
    reward_cache = {}
    for i, ex in enumerate(raw_data):
        reward_cache[i] = compute_reward(ex)
    rvals = list(reward_cache.values())
    print(f"  Mean reward: {np.mean(rvals):.3f}  "
          f"std: {np.std(rvals):.3f}  "
          f"min: {np.min(rvals):.3f}  max: {np.max(rvals):.3f}")

    print("\nTokenising...")
    data_with_ids = []
    skip = defaultdict(int)
    raw_indices   = []
    for orig_idx, ex in enumerate(raw_data):
        try:
            datum = conversation_to_datum(
                ex["messages"], renderer,
                max_length=args.max_length,
                train_on_what=renderers.TrainOnWhat.ALL_ASSISTANT_MESSAGES,
            )
            data_with_ids.append((datum, ex["task_id"], ex, orig_idx))
        except Exception as e:
            skip["too_long" if "length" in str(e).lower() else "error"] += 1

    task_counts = {name: sum(1 for _, tid, _, _ in data_with_ids if tid == i)
                   for i, name in TASK_NAMES.items()}
    print(f"  Ready: {len(data_with_ids)} | skipped: {dict(skip)}")
    print(f"  Per-task: {task_counts}")

    if not data_with_ids:
        raise RuntimeError("No training data after filtering!")

    print(f"\nCreating LoRA client (rank={args.rank})...")
    sc = tinker.ServiceClient()
    if args.resume_from:
        print(f"  Resuming: {args.resume_from}")
        tc = sc.create_training_client_from_state_with_optimizer(args.resume_from)
    else:
        tc = sc.create_lora_training_client(base_model=MODEL, rank=args.rank)
    print("  Ready\n")

    loader       = ShuffledDataLoader(data_with_ids, args.batch_size, seed=args.seed)
    lr_adapter   = TaskLRAdapter(list(TASK_NAMES.keys()),
                                 window=args.task_lr_window,
                                 boost=args.task_lr_boost)

    task_loss_accum  = defaultdict(float)
    task_loss_counts = defaultdict(int)
    reward_accum     = defaultdict(float)
    reward_counts    = defaultdict(int)
    intermediate_checkpoints = []

    _last_lr  = {}   # task_id → last lr used
    _adam_cache = {}

    def get_adam(lr):
        if lr not in _adam_cache:
            _adam_cache[lr] = types.AdamParams(
                learning_rate=lr, beta1=0.9, beta2=0.95, eps=1e-8)
        return _adam_cache[lr]

    print(f"Training {args.num_steps} steps "
          f"(SFT 0-{args.sft_steps-1} | RL-SFT {args.sft_steps}-{args.num_steps-1})...")

    for step in range(args.num_steps):

        in_rl_phase = step >= args.sft_steps
        base_lr = args.lr * (args.rl_lr_scale if in_rl_phase else 1.0)
        lr      = get_lr(step, base_lr, args.warmup_steps, args.num_steps)

        batch = loader.next_batch()

        raw_rewards = [reward_cache[orig_idx] for _, _, _, orig_idx in batch]
        task_ids    = [b[1] for b in batch]

        if args.use_task_lr:
            for tid, r in zip(task_ids, raw_rewards):
                lr_adapter.update(tid, r)

        if args.use_reward_weights:
            norm_w = normalise_rewards(raw_rewards)
        else:
            norm_w = [1.0] * len(batch)

        if args.use_batch_contrast and in_rl_phase:
            contrast_scales = batch_contrastive_scale(raw_rewards,
                                                       clip=args.contrast_clip)
        else:
            contrast_scales = [1.0] * len(batch)

        combined_scales = [nw * cs for nw, cs in zip(norm_w, contrast_scales)]

        if args.use_task_lr:
            task_counter = Counter(task_ids)
            dominant_task = task_counter.most_common(1)[0][0]
            task_scale    = lr_adapter.scale(dominant_task)
            step_lr = lr * task_scale
        else:
            step_lr = lr

        adam_params = get_adam(round(step_lr, 10))

        datums = []
        for (datum, _, _, _), scale in zip(batch, combined_scales):
            if abs(scale - 1.0) > 1e-6:
                datum = apply_weight_scale(datum, scale)
            datums.append(datum)

        fwd  = tc.forward_backward(datums, loss_fn="cross_entropy")
        tc.optim_step(adam_params).result()
        res  = fwd.result()

        lp = np.concatenate([o["logprobs"].tolist() for o in res.loss_fn_outputs])
        w  = np.concatenate([d.loss_fn_inputs["weights"].to_numpy() for d in datums])
        loss = -np.dot(lp, w) / max(w.sum(), 1)

        for tid, r in zip(task_ids, raw_rewards):
            task_loss_accum[tid]  += loss / len(task_ids)
            task_loss_counts[tid] += 1
            reward_accum[tid]     += r
            reward_counts[tid]    += 1

        if (step + 1) % args.log_interval == 0 or step == 0:
            phase = "RL-SFT" if in_rl_phase else "SFT  "
            task_loss_str = []
            task_rew_str  = []
            for tid in range(3):
                if task_loss_counts[tid] > 0:
                    avg_l = task_loss_accum[tid] / task_loss_counts[tid]
                    avg_r = reward_accum[tid] / reward_counts[tid]
                    task_loss_str.append(f"{TASK_NAMES[tid]}: {avg_l:.3f}")
                    task_rew_str.append( f"{TASK_NAMES[tid]}: {avg_r:.3f}")
                    task_loss_accum[tid] = task_loss_counts[tid] = 0
                    reward_accum[tid]    = reward_counts[tid]    = 0
            if args.use_task_lr:
                lr_scales = " ".join(
                    f"{TASK_NAMES[tid]}: ×{lr_adapter.scale(tid):.2f}"
                    for tid in range(3))
                lr_info = f" | LR-scales [{lr_scales}]"
            else:
                lr_info = ""
            print(f"  [{phase}] Step {step+1:4d}/{args.num_steps}"
                  f" | LR: {step_lr:.2e}"
                  f" | Loss [{' | '.join(task_loss_str)}]"
                  f" | Reward [{' | '.join(task_rew_str)}]"
                  f" | Epoch: {loader.epoch}"
                  + lr_info)

        if (step + 1) % args.save_every == 0 and (step + 1) < args.num_steps:
            phase_tag = "rl-sft" if in_rl_phase else "sft"
            ckpt_name = f"{args.checkpoint_name}_step{step+1}_{phase_tag}"
            print(f"\n  Saving: {ckpt_name}")
            ckpt = tc.save_weights_for_sampler(
                ckpt_name, ttl_seconds=args.intermediate_ttl).result()
            print(f"  Saved: {ckpt.path}")
            intermediate_checkpoints.append(
                {"step": step+1, "path": ckpt.path, "phase": phase_tag})

    print(f"\nSaving '{args.checkpoint_name}'...")
    ckpt  = tc.save_weights_for_sampler(name=args.checkpoint_name).result()
    state = tc.save_state(f"{args.checkpoint_name}_state").result()
    print(f"  Checkpoint: {ckpt.path}")
    print(f"  State:      {state.path}")

    if not args.no_publish:
        sc.create_rest_client().publish_checkpoint_from_tinker_path(ckpt.path).result()
        print("  Published!")

    info = {
        "checkpoint_path": ckpt.path,
        "state_path":      state.path,
        "intermediate_checkpoints": intermediate_checkpoints,
        "base_model":      MODEL,
        "training": {
            "num_steps":          args.num_steps,
            "sft_steps":          args.sft_steps,
            "batch_size":         args.batch_size,
            "learning_rate":      args.lr,
            "rl_lr_scale":        args.rl_lr_scale,
            "warmup_steps":       args.warmup_steps,
            "lora_rank":          args.rank,
            "max_length":         args.max_length,
            "use_reward_weights": args.use_reward_weights,
            "use_batch_contrast": args.use_batch_contrast,
            "contrast_clip":      args.contrast_clip,
            "use_task_lr":        args.use_task_lr,
            "task_lr_boost":      args.task_lr_boost,
            "task_lr_window":     args.task_lr_window,
            "data_path":          args.data_path,
            "total_examples":     len(data_with_ids),
            "per_task":           task_counts,
            "resumed_from":       args.resume_from,
        },
    }
    info_path = os.path.join(EVAL_DIR, "checkpoint_info.json")
    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)
    print(f"  Info → {info_path}")
    print(f"\nEvaluate with:")
    print(f'  python evaluation/eval_all.py --checkpoint_path "{ckpt.path}" --base_model {MODEL}')


if __name__ == "__main__":
    main()

Overwriting evaluation/train_and_publish.py


In [ ]:
!python evaluation/train_and_publish.py \
    --data_path evaluation/training_data.jsonl \
    --num_steps 5000 \
    --sft_steps 800 \
    --lr 1e-4 \
    --rl_lr_scale 0.3 \
    --rank 32 \
    --warmup_steps 200 \
    --contrast_clip 0.3 \
    --task_lr_boost 0.25 \
    --save_every 500 \
    --checkpoint_name "llama8b_v3"

Model:              meta-llama/Llama-3.1-8B
Steps:              5000  (SFT warm-start: 800)
Batch size:         4
LR:                 0.0001  (phase-2 × 0.3)
LoRA rank:          32
Reward weights:     True
Batch contrast:     True  (clip=0.3)
Task adaptive LR:   True  (boost=0.25)
Renderer: role_colon

Loading data...
  29998 examples
  GSM8K: 9999
  IFEval: 10000
  Code: 9999

Pre-computing rewards...
<unknown>:5: SyntaxWarning: invalid escape sequence '\w'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:7: SyntaxWarning: invalid escape sequence '\d'
<unknown>:24: SyntaxWarning: invalid escape sequence '\{'
<unknown>:13: SyntaxWarning: invalid escape sequence '\W'
<unknown>:22: SyntaxWarning: invalid escape sequence '\w'
<unknown>:9: SyntaxWarning: invalid escape sequence '\s'
  Mean reward: 0.547  std: 0.416  min: 0.000  max: 1.000

Tokenising...
  Ready: 29998 | skipped: {}
  Per-task: {'GSM8K': 9999, 'IFEval': 10000, 'Code': 9999}

Creating LoRA client (rank=32).

In [ ]:
import json

with open("/content/project/evaluation/checkpoint_info.json") as f:
    info = json.load(f)

ckpt = info["checkpoint_path"]
print(f"checkpoint: {ckpt}")

checkpoint: tinker://097a20e5-73a5-594e-9d73-57b333025f1e:train:0/sampler_weights/final_8B_v2


In [ ]:
!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step500_sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step1000_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step1500_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step2000_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step2500_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step3000_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step3500_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step4000_rl-sft" --base_model meta-llama/Llama-3.1-8B

!python evaluation/eval_all.py --checkpoint_path "tinker://65e2f215-29d7-5e56-91ba-64deba4c867f:train:0/sampler_weights/llama8b_v3_step4500_rl-sft" --base_model meta-llama/Llama-3.1-8B

AttributeError: module 'tinker' has no attribute 'list_artifacts'